In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

BASE_PATH = '/content/drive/Shareddrives/Sieci_Neuronowe/Projekt_1'

with open(f'{BASE_PATH}/imagenet_112x112.json', 'r', encoding='utf-8') as f:
    d = json.load(f)

h = d['history']
epochs    = [x['epoch']     for x in h]
train_acc = [x['train_acc'] for x in h]
test_acc  = [x['test_acc']  for x in h]
loss      = [x['loss']      for x in h]

print('Dane załadowane!')
print(f"Dataset   : {d['dataset']}")
print(f"img_size  : {d['img_size']}x{d['img_size']}  (patch={d['patch_size']} → {d['num_tokens']} tokenów)")
print(f"Parametry : {d['num_params']:,}")
print(f"Epoki     : {d['epochs']}")
print(f"Best acc  : {d['best_test_acc']:.2f}%  @ epoch {d['best_epoch']}")
print(f"Final acc : {d['final_test_acc']:.2f}%")
print(f"Czas      : {d['train_time_s']/3600:.2f}h")

## Konfiguracja modelu

| Parametr | Wartość |
|---|---|
| Architektura | VisionTransformer |
| Dataset | Imagenette2-320 (10 klas) |
| Rozdzielczość | **112×112** (vs 32×32 w poprzednich) |
| Patch size | **8** → (112/8)² = **196 tokenów** |
| embed_dim | 256 |
| num_heads | 8 |
| Bloki Nx | 6 |
| mlp_dim | 1024 |
| Dropout | 0.2 |
| Optimizer | **AdamW** (weight_decay=0.05) |
| Scheduler | **CosineAnnealingLR** (eta_min=1e-6) |
| Label smoothing | **0.1** |
| Epoki | 150 |
| Parametry | ~4.79M |

## 1. Krzywa Treningu — Accuracy

In [ ]:
best_idx = int(np.argmax(test_acc))

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(epochs, train_acc, 'o-', label='Train Accuracy', color='#2E86AB', lw=2, markersize=5)
ax.plot(epochs, test_acc,  's-', label='Test Accuracy',  color='#A23B72', lw=2, markersize=5)

ax.scatter([epochs[best_idx]], [test_acc[best_idx]],
           s=250, color='gold', edgecolor='black', zorder=5,
           label=f'Best: {test_acc[best_idx]:.2f}% (ep {epochs[best_idx]})')

# Linie referencyjne z poprzednich eksperymentów
ax.axhline(d['vs_32x32_150ep'], color='#4ECDC4', linestyle='--', lw=1.5,
           label=f'32×32 150ep ({d["vs_32x32_150ep"]:.2f}%)')
ax.axhline(d['vs_32x32_10ep'],  color='#FF6B6B', linestyle=':',  lw=1.5,
           label=f'32×32 10ep  ({d["vs_32x32_10ep"]:.2f}%)')

ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('ViT @ 112×112 — 150 Epochs — Accuracy (Imagenette2-320)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/112x112_01_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Krzywa Treningu — Loss

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(epochs, loss, '^-', color='#F18F01', lw=2, markersize=5, label='Training Loss')
ax.axvline(epochs[best_idx], color='#A23B72', linestyle='--', lw=1.5,
           label=f'Best test acc @ ep {epochs[best_idx]}')

ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('ViT @ 112×112 — 150 Epochs — Loss (Imagenette2-320)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/112x112_02_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Porównanie Rozdzielczości: 32×32 vs 112×112

In [ ]:
configs = ['ViT\n32×32\n10 ep', 'ViT\n32×32\n150 ep', 'ViT\n112×112\n150 ep']
accs    = [d['vs_32x32_10ep'], d['vs_32x32_150ep'], d['best_test_acc']]
colors  = ['#FF6B6B', '#4ECDC4', '#1B998B']

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(configs, accs, color=colors, alpha=0.85, edgecolor='black', width=0.5)
ax.set_ylabel('Test Accuracy (%)', fontsize=11)
ax.set_title('Imagenette2-320: Wpływ Rozdzielczości i Liczby Epok',
             fontsize=12, fontweight='bold')
ax.set_ylim(60, 78)
ax.grid(axis='y', alpha=0.3)

for i, v in enumerate(accs):
    ax.text(i, v + 0.3, f'{v:.2f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Strzałka pokazująca zysk
gain = d['best_test_acc'] - d['vs_32x32_150ep']
ax.annotate('', xy=(2, d['best_test_acc'] - 0.3), xytext=(1, d['vs_32x32_150ep'] + 0.3),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(1.5, (d['best_test_acc'] + d['vs_32x32_150ep'])/2,
        f'+{gain:.2f}%', ha='center', va='center', fontsize=11,
        fontweight='bold', color='darkgreen',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='darkgreen'))

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/112x112_03_resolution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Zysk 112×112 vs 32×32 (150ep): +{gain:.2f}%')
print(f'Zysk 112×112 vs 32×32 (10ep):  +{d["best_test_acc"] - d["vs_32x32_10ep"]:.2f}%')

## 4. Milestones co 10 epok — Accuracy i Loss

In [ ]:
# Historia jest zapisana co 5 epok — weź co 10 (parzyste milestony)
ms = [x for x in h if x['epoch'] % 10 == 0]
ep_m = [x['epoch'] for x in ms]
ta_m = [x['train_acc'] for x in ms]
te_m = [x['test_acc']  for x in ms]
lo_m = [x['loss']      for x in ms]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ViT 112×112 — Milestones co 10 epok', fontsize=13, fontweight='bold')

x = np.arange(len(ep_m))
w = 0.4

# Accuracy
axes[0].bar(x - w/2, ta_m, w, label='Train Acc', color='#2E86AB', alpha=0.85, edgecolor='black')
axes[0].bar(x + w/2, te_m, w, label='Test Acc',  color='#A23B72', alpha=0.85, edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'ep {e}' for e in ep_m], rotation=45)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

# Loss
axes[1].bar(x, lo_m, color='#F18F01', alpha=0.85, edgecolor='black', label='Loss')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'ep {e}' for e in ep_m], rotation=45)
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/112x112_04_milestones.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Podsumowanie

In [ ]:
print('='*65)
print('PODSUMOWANIE — ViT @ 112×112 (Imagenette2-320)')
print('='*65)
print(f"\nArchitektura:")
print(f"  img_size   = {d['img_size']}×{d['img_size']}")
print(f"  patch_size = {d['patch_size']}  →  {d['num_tokens']} tokenów")
for k, v in d['config'].items():
    print(f"  {k:<20} = {v}")
print(f"  num_params           = {d['num_params']:,}")
print(f"\nWyniki:")
print(f"  Best test acc  : {d['best_test_acc']:.2f}%  @ epoch {d['best_epoch']}")
print(f"  Final test acc : {d['final_test_acc']:.2f}%")
print(f"  Final train acc: {d['final_train_acc']:.2f}%")
print(f"  Overfitting    : {d['final_train_acc'] - d['final_test_acc']:.2f}%")
print(f"  Czas treningu  : {d['train_time_s']/3600:.2f}h  ({d['time_per_epoch']:.1f}s/epoka)")
print(f"\nPorównanie rozdzielczości (Imagenette2-320):")
print(f"  32×32  10 ep  : {d['vs_32x32_10ep']:.2f}%")
print(f"  32×32 150 ep  : {d['vs_32x32_150ep']:.2f}%")
print(f"  112×112 150ep : {d['best_test_acc']:.2f}%  ← ta symulacja")
print(f"  Zysk vs 32×32 : +{d['best_test_acc'] - d['vs_32x32_150ep']:.2f}%")
print('='*65)